# Country–Year Non-Communicable Disease Mortality, Expected Deaths, and Environmental Covariates for the East African Community, 2000–2019: Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. 

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.jqaw-53vv/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Authors: {getattr(metadata, 'author', 'N/A')}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Keywords: {getattr(metadata, 'keywords', 'N/A')}")

## 2. Data Overview

Review available record sets and fields in the dataset. All references use Croissant entity `@id`s for reproducibility.

In [ ]:
# List all available record sets (and their @id)
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"- Record set '@id': {rs['@id']}, Name: {rs.get('name','(none)')}")
    # List fields in this record set
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("    Fields:")
    for fld in fields:
        if isinstance(fld, dict):
            print(f"      - Field '@id': {fld.get('@id', '')}, Name: {fld.get('name','(none)')}, DataType: {fld.get('dataType','(none)')}")
        else:
            print(f"      - Field: {fld}")
    print()
# Save one record set id for later extraction
if len(record_sets) > 0:
    example_record_set_id = record_sets[0]['@id']
else:
    example_record_set_id = None

## 3. Data Extraction

Load data from the primary record set into a DataFrame for analysis. Use the record set and field `@id`s listed above. You may adapt the record set ID and field IDs as needed for your analysis.

In [ ]:
# Replace the list below with the discovered record set @ids from the previous cell if extracting multiple tables
record_set_ids = [example_record_set_id] if example_record_set_id else []
dataframes = {}

for rs_id in record_set_ids:
    print(f"Extracting records from record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded dataframe shape: {df.shape}\nColumns: {df.columns.tolist()}")
    # Show a preview
    display(df.head(3))

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps: filter country-years, normalize numeric fields, group by attributes, etc.

Below, we demonstrate filtering and normalization using a numeric field. Update `<numeric_field_id>` and `<group_field_id>` with actual `@id` values from your overview.

In [ ]:
import numpy as np

# For demonstration, select a numeric field based on the columns discovered above
# You should change these IDs to those found in your dataset for deeper analysis
record_set_id = example_record_set_id
if not record_set_id:
    print("No available record set to perform EDA.")
else:
    df = dataframes[record_set_id]

    # Try to pick a likely numeric field for demo purposes
    # This logic adapts if columns are named with common health data conventions
    preferred_fields = ['expected_deaths', 'observed_deaths', 'mortality_rate']
    found_numeric = None
    for f in preferred_fields:
        if f in df.columns:
            found_numeric = f
            break
    if not found_numeric:
        numeric_field = df.select_dtypes(include=np.number).columns[0] if len(df.select_dtypes(include=np.number).columns) else df.columns[0]
    else:
        numeric_field = found_numeric
    print(f"Selected numeric field for EDA: {numeric_field}")

    # Set threshold for demonstration (tune this for your own exploration)
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]

    print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} records\n")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by a likely field, e.g., 'country' or similar
    group_fields = [c for c in df.columns if any(x in c.lower() for x in ['country', 'region', 'group'])]
    if group_fields:
        group_field = group_fields[0]
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped mean (numeric only) values by '{group_field}':")
        display(grouped_df.head())
    else:
        print("No obvious group field found for grouping.")

## 5. Visualization

Visualize the distribution of the selected numeric field and its variation over a grouping variable, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Histogram of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by country/group if possible
    if group_fields:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

We demonstrated how to use `mlcroissant` to load the FAIR\(^2\) EAC NCD dataset, explored its structure and contents via Croissant `@id`s, and performed initial data wrangling and visualization.

Further analysis—such as multivariate modelling, temporal trend examination, and geospatial exploration—can build on these steps using the harmonized data records and metadata from Croissant.